# Session 9: Synthetic Data Generation and RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow, and use it to evaluate and iterate on a RAG pipeline with LangSmith!

**Learning Objectives:**
- Understand Ragas' knowledge graph-based synthetic data generation workflow
- Generate synthetic test sets with different query synthesizer types
- Load synthetic data into LangSmith for evaluation
- Evaluate a RAG chain using LangSmith evaluators
- Iterate on RAG pipeline parameters and measure the impact

## Table of Contents:

- **Breakout Room #1:** Synthetic Data Generation with Ragas
  - Task 1: Dependencies and API Keys
  - Task 2: Data Preparation and Knowledge Graph Construction
  - Task 3: Generating Synthetic Test Data
  - Question #1 & Question #2
  - 🏗️ Activity #1: Custom Query Distribution

- **Breakout Room #2:** RAG Evaluation with LangSmith
  - Task 4: LangSmith Dataset Setup
  - Task 5: Building a Basic RAG Chain
  - Task 6: Evaluating with LangSmith
  - Task 7: Modifying the Pipeline and Re-Evaluating
  - Question #3 & Question #4
  - 🏗️ Activity #2: Analyze Evaluation Results

---
# 🤝 Breakout Room #1
## Synthetic Data Generation with Ragas

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/denismcphillips/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/denismcphillips/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using two complementary guides — a Health & Wellness Guide covering exercise, nutrition, sleep, and stress management, and a Mental Health & Psychology Handbook covering mental health conditions, therapeutic approaches, resilience, and daily mental health practices. The topical overlap between documents helps RAGAS build rich cross-document relationships in the knowledge graph.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader("data/", glob="*.txt", loader_cls=TextLoader)
docs = loader.load()
print(f"Loaded {len(docs)} documents: {[d.metadata['source'] for d in docs]}")

Loaded 2 documents: ['data/MentalHealthGuide.txt', 'data/HealthWellnessGuide.txt']


### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 2, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/16 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 9, relationships: 14)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 9, relationships: 14)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

## ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### Answer:
- `SingleHopSpecificQuerySynthesizer` only requires pulling data from a single document to answer a fact-based query.
- `MultiHopAbstractQuerySynthesizer` requires using reasoning to answer a more open-ended query, pulling the answer from multiple documents.
- `MultiHopSpecificQuerySynthesizer` pulls data from multiple documents to answer a multifaceted, fact-based query.

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is mental health and how does it affect p...,[The Mental Health and Psychology Handbook A P...,"Mental health encompasses our emotional, psych...",single_hop_specifc_query_synthesizer
1,What is DBT?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Dialectical Behavior Therapy (DBT) is original...,single_hop_specifc_query_synthesizer
2,What is dementia?,[Write letters to or from your future self Jou...,Dementia is mentioned in the context as a cond...,single_hop_specifc_query_synthesizer
3,How does social media impact mental health and...,[social interactions How to set and maintain b...,Social media can affect mental health by lower...,single_hop_specifc_query_synthesizer
4,What is discussed in Chapter 11?,[PART 2: NUTRITION AND DIET Chapter 4: Fundame...,"Chapter 11 covers stress reduction techniques,...",single_hop_specifc_query_synthesizer
5,How social connection and setting boundaries h...,[<1-hop>\n\nWrite letters to or from your futu...,The context explains that strong social connec...,multi_hop_abstract_query_synthesizer
6,how can personal wellness and healthy habits l...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,The personal wellness guide emphasizes that re...,multi_hop_abstract_query_synthesizer
7,How does gut-brain axis and its role in mood r...,[<1-hop>\n\nThe Mental Health and Psychology H...,The gut-brain axis plays a significant role in...,multi_hop_abstract_query_synthesizer
8,How can understanding and applying Cognitive B...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,Understanding and applying Cognitive Behaviora...,multi_hop_specific_query_synthesizer
9,How can combining CBT techniques with mindfuln...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,The context explains that Cognitive Behavioral...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/18 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Wht is the role of the World Health Organization?,[The Mental Health and Psychology Handbook A P...,"According to the context, the World Health Org...",single_hop_specifc_query_synthesizer
1,What is MBSR?,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Mindfulness-Based Stress Reduction (MBSR) is a...,single_hop_specifc_query_synthesizer
2,How do B vitamins help in maintaining good men...,[Write letters to or from your future self Jou...,"B vitamins are found in whole grains, eggs, an...",single_hop_specifc_query_synthesizer
3,How do psychologists contribute to managing di...,[social interactions How to set and maintain b...,Psychologists specialize in therapy and assess...,single_hop_specifc_query_synthesizer
4,How do practices related to sleep hygiene and ...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,"Practicing good sleep hygiene, such as maintai...",multi_hop_abstract_query_synthesizer
5,How can mindfulness and meditation practices i...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,Mindfulness and meditation practices can incor...,multi_hop_abstract_query_synthesizer
6,How can meal planning and preparation help me ...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Meal planning and preparation are essential fo...,multi_hop_abstract_query_synthesizer
7,"How can u use the habit loop: cue, routine, re...",[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,"To use the habit loop—cue, routine, reward—to ...",multi_hop_abstract_query_synthesizer
8,"How do the therapeutic approaches in PART 2, s...",[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,"The therapeutic approaches outlined in PART 2,...",multi_hop_specific_query_synthesizer
9,How do Chapters 4 and 9 collectively emphasize...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Chapter 4 discusses the fundamentals of health...,multi_hop_specific_query_synthesizer


## ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### Answer:
The unrolled approach is more customizable, allowing us to specify query distribution, transformations, and similarity metric between the embeddings. However, this is more complex and time-consuming to set up than the abstracted approach, which only requires providing an llm and embedding model (at the cost of less customization than the unrolled approach). Additionally, the unrolled approach allows us to save and load the knowledge graph for use across multiple generation runs, while the abstracted approach rebuilds the graph each time (adding costs and latency). Further, the abstracted approach is somewhat of a "black box" where we are unable to inspect nodes and relationships in the knowledge graph, which we are able to do with the unrolled approach.

I would choose the abstracted approach when looking to generate synthetic data quickly, while I would use the unrolled approach when I want to tweak query distribution or use a different similarity metric.


---
## 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

### Requirements:
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [16]:
### YOUR CODE HERE ###

# Define a custom query distribution with different weights
custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.4),
]

# Generate a new test set and compare with the default
custom_testset = generator.generate(testset_size=10, query_distribution=custom_query_distribution)
custom_testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,like what is mental health and why is it impor...,[The Mental Health and Psychology Handbook A P...,"Mental health encompasses our emotional, psych...",single_hop_specifc_query_synthesizer
1,What is Mindfulness-Based Stress Reduction and...,[PART 2: THERAPEUTIC APPROACHES Chapter 4: Cog...,Mindfulness-Based Stress Reduction (MBSR) is a...,single_hop_specifc_query_synthesizer
2,How does the mind-body connection relate to mi...,[<1-hop>\n\nThe Mental Health and Psychology H...,The mind-body connection demonstrates that men...,multi_hop_abstract_query_synthesizer
3,"so tell me how factors like biological, family...",[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that mental health is aff...,multi_hop_abstract_query_synthesizer
4,How can establishing a relaxing bedtime routin...,[<1-hop>\n\nPART 3: SLEEP AND RECOVERY Chapter...,"Establishing a relaxing bedtime routine, such ...",multi_hop_abstract_query_synthesizer
5,How does the mnd-body connecion and mindfulnes...,[<1-hop>\n\nThe Mental Health and Psychology H...,The context explains that the mind-body connec...,multi_hop_abstract_query_synthesizer
6,whats PART 2 and PART 5 about and how do they ...,[<1-hop>\n\nPART 2: THERAPEUTIC APPROACHES Cha...,PART 2 discusses therapeutic approaches like C...,multi_hop_specific_query_synthesizer
7,How do Chapters 6 and 9 of the Personal Wellne...,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,Chapter 6 highlights that water is essential f...,multi_hop_specific_query_synthesizer
8,Chapter 6 and 17 how they help wellness?,[<1-hop>\n\nThe Personal Wellness Guide A Comp...,"Chapter 6 talks about healthy eating, hydratio...",multi_hop_specific_query_synthesizer
9,how build habits from chapter 8 and 13 help we...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,chapter 13 talks about building habits by unde...,multi_hop_specific_query_synthesizer


##### Why these weights?

This puts more weight toward multi-hop queries (`0.4` each for abstract and specific) and reduced single-hop to `0.2`, compared to the default `0.5 / 0.25 / 0.25`. This stress-tests the system's ability to reason across multiple documents, since real-world users often ask questions that require information from multiple sources. While smaller, we must still include single-hop queries to maintain baseline factual recall coverage.

The default distribution produced 50% single-hop queries (straightforward, fact-lookup questions from a single document). Our custom distribution generates 80% multi-hop queries, which tend to be longer, reference multiple topics, and require cross-document reasoning. This new weight distribution creates a more challenging data set that focuses on evaluating retrieval and synthesis.

We'll need to provide our LangSmith API key, and set tracing to "true".

---
# 🤝 Breakout Room #2
## RAG Evaluation with LangSmith

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client
import uuid

client = Client()

dataset_name = f"Use Case Synthetic Data - AIE9 - {uuid.uuid4()}"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="use_case_rag"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What are some recommended exercises for lower back pain?"})

'Recommended exercises for lower back pain include:\n\n- Cat-Cow Stretch: Start on hands and knees, alternate between arching your back up (cat) and letting it sag down (cow). Do 10-15 repetitions.\n- Bird Dog: From hands and knees, extend opposite arm and leg while keeping your core engaged. Hold for 5 seconds, then switch sides. Do 10 repetitions per side.\n- Partial Crunches: Lie on your back with knees bent, cross arms over chest, tighten stomach muscles and raise shoulders off floor. Hold briefly, then lower. Do 8-12 repetitions.\n- Knee-to-Chest Stretch: Lie on your back, pull one knee toward your chest while keeping the other foot flat. Hold for 15-30 seconds, then switch legs.\n- Pelvic Tilts: Lie on your back with knees bent, flatten your back against the floor by tightening abs and tilting pelvis up slightly. Hold for 10 seconds, repeat 8-12 times.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from openevals.llm import create_llm_as_judge
from langsmith.evaluation import evaluate

# 1. QA Correctness (replaces LangChainStringEvaluator("qa"))
qa_evaluator = create_llm_as_judge(
    prompt="You are evaluating a QA system. Given the input, assess whether the prediction is correct.\n\nInput: {inputs}\nPrediction: {outputs}\nReference answer: {reference_outputs}\n\nIs the prediction correct? Return 1 if correct, 0 if incorrect.",
    feedback_key="qa",
    model="openai:gpt-4o" ,  # pass your LangChain chat model directly
)

# 2. Labeled Helpfulness (replaces LangChainStringEvaluator("labeled_criteria"))
labeled_helpfulness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "helpfulness: Is this submission helpful to the user, "
        "taking into account the correct reference answer?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n"
        "Reference answer: {reference_outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="helpfulness",
    model="openai:gpt-4o" ,
)

# 3. Dopeness (replaces LangChainStringEvaluator("criteria"))
dopeness_evaluator = create_llm_as_judge(
    prompt=(
        "You are assessing a submission based on the following criterion:\n\n"
        "dopeness: Is this response dope, lit, cool, or is it just a generic response?\n\n"
        "Input: {inputs}\n"
        "Submission: {outputs}\n\n"
        "Does the submission meet the criterion? Return 1 if yes, 0 if no."
    ),
    feedback_key="dopeness",
    model="openai:gpt-4o" ,
)

> **Describe what each evaluator is evaluating:**
>
> - `qa_evaluator`: correctness of an output
> - `labeled_helpfulness_evaluator`: how helpful the output is to the user
> - `dopeness_evaluator`: how dope / not generic the output is

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'cooked-join-12' at:
https://smith.langchain.com/o/9602aa7a-2a36-48c1-af85-1ca60c09239b/datasets/276b2ac9-1578-4934-bd94-2ffa349c6242/compare?selectedSessions=5fef6491-97c1-40b6-90b1-741b92d8cc96




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How do the principles of habit formation discu...,The principles of habit formation discussed in...,None,The principles of habit formation in Chapter 1...,True,True,True,7.282992,c07d8168-afde-4bca-a124-496f16a95bd0,019c689a-7b34-71f2-bada-3834dda637b2
1,Can you tell me how Chapter 3 and Chapter 16 a...,"Based on the provided context, Chapter 3 (""Bui...",None,"Chapter 3 discusses building healthy habits, i...",True,True,True,3.257811,46175351-46dc-4ff3-9acd-7a61ae3d9e16,019c689a-b951-7712-8d7e-3f5859707627
2,How do Chapters 4 and 9 collectively emphasize...,I don't know.,None,Chapter 4 discusses the fundamentals of health...,False,False,False,2.226178,4e7c8cbd-fb14-4195-a86c-69bc9c43a0c6,019c689a-f60a-7400-b781-3932da47e0a7
3,"How do the therapeutic approaches in PART 2, s...",Based on the context provided:\n\nCBT (Cogniti...,None,"The therapeutic approaches outlined in PART 2,...",True,True,True,3.935506,ccd8670f-ec2c-49b6-b143-2d65594080d5,019c689b-1d22-7d70-a0a6-806528977844
4,"How can u use the habit loop: cue, routine, re...","Based on the provided context, you can use the...",None,"To use the habit loop—cue, routine, reward—to ...",True,True,False,4.094976,7fb5c52a-6a9f-4af1-abcb-3599883a52e3,019c689b-4e82-7e72-865a-7be3745401d7
5,How can meal planning and preparation help me ...,Meal planning and preparation can help you cre...,None,Meal planning and preparation are essential fo...,True,True,True,2.968504,b5806079-4d65-4b1d-9837-a11f2a5929dd,019c689b-91b7-7b60-aace-118bdd7a5e42
6,How can mindfulness and meditation practices i...,Based on the context provided:\n\nMindfulness ...,None,Mindfulness and meditation practices can incor...,True,True,True,4.705983,65121dbb-7fdb-4e7f-9a84-b54e498e62d9,019c689b-c352-7dc1-817e-4be5af793026
7,How do practices related to sleep hygiene and ...,Practices related to sleep hygiene and hydrati...,None,"Practicing good sleep hygiene, such as maintai...",True,True,False,4.164898,36cf455e-572f-4623-85b0-994f4f16342a,019c689c-290b-7320-a5a8-f3c77a11f18b
8,How do psychologists contribute to managing di...,I don't know.,None,Psychologists specialize in therapy and assess...,False,False,False,0.713935,5cadf8cd-c2d2-40f6-bf3f-fb3dd779e808,019c689c-6551-7273-a45d-b9eec3433c53
9,How do B vitamins help in maintaining good men...,B vitamins help in maintaining good mental hea...,None,"B vitamins are found in whole grains, eggs, an...",True,True,False,0.775945,1404ae5a-f868-4e84-b478-5cf15e57dc77,019c689c-8a1d-71a1-9f5d-d39c9bb8d43b


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

## ❓ Question #3:

Why would modifying our chunk size modify the performance of our application?

##### Answer:
Modifying chunk size modifies the performance of our application because this determines how many chunks are created - having fewer chunks means longer strings of text, while more chunks means shorter strings of text. Both of these can impact similarity scores: shorter strings may split a relevant piece of information across two chunks but are more likely to have higher similarity scores; larger chunks may contain more topics and have lower similarity scores but are less likely to have relevant information split.

In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

## ❓ Question #4:

Why would modifying our embedding model modify the performance of our application?

##### Answer:
Choosing a more powerful embedding model would likely lead to better retrieval (and better responses) due to more dimensions for each vector/higher likelihood of capturing more nuanced similarities, but this comes with higher costs and latency. Inversely, less powerful models would have lower costs/latency but may not be as successful with retrieval.

In [34]:
from langchain_qdrant import QdrantVectorStore

vectorstore = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
dopeness_rag_chain.invoke({"question" : "How can I improve my sleep quality?"})

"Alright, let's level up your sleep game with some next-level tips straight from the ultimate sleep playbook:\n\n1. **Own your schedule like a boss** — hit the sack and rise at the same time every single day, weekends included. Consistency is the secret sauce.\n\n2. **Craft a bedtime ritual that chills you out** — dive into a good book, do gentle stretches, or soak in a warm bath. This signals your brain it’s go-time for dreamland.\n\n3. **Turn your bedroom into a sleep fortress** — keep it cool between 65-68°F (that’s 18-20 Celsius if you’re feeling fancy), pitch-black with blackout curtains or a sleep mask, and hush-hush with white noise machines or earplugs.\n\n4. **Screen off, lights dimmed** — ditch your phone and any screens 1-2 hours before you hit the hay. Blue light is a sleep thief.\n\n5. **Cut caffeine after 2 PM** — no more jitter juice in the afternoon; your body needs to mellow out.\n\n6. **Get moving, but not too close to bedtime** — regular exercise boosts sleep quality

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'slight-color-54' at:
https://smith.langchain.com/o/9602aa7a-2a36-48c1-af85-1ca60c09239b/datasets/276b2ac9-1578-4934-bd94-2ffa349c6242/compare?selectedSessions=e43be04c-bb4b-48e6-9e12-53cf47062d4e




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.qa,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How do the principles of habit formation discu...,"Alright, let’s dive deep into the gnarly syner...",None,The principles of habit formation in Chapter 1...,True,True,True,8.199536,c07d8168-afde-4bca-a124-496f16a95bd0,019c689d-cc29-77d2-bec2-94fe39415909
1,Can you tell me how Chapter 3 and Chapter 16 a...,"Yo, let’s vibe with this connection like a wel...",None,"Chapter 3 discusses building healthy habits, i...",True,True,True,5.047401,46175351-46dc-4ff3-9acd-7a61ae3d9e16,019c689e-1001-7b93-997d-c9012fed605c
2,How do Chapters 4 and 9 collectively emphasize...,"Yo, here’s the deal: the context doesn’t spill...",None,Chapter 4 discusses the fundamentals of health...,False,False,True,2.776006,4e7c8cbd-fb14-4195-a86c-69bc9c43a0c6,019c689e-4863-7ad2-9aa6-c1be4573ccc4
3,"How do the therapeutic approaches in PART 2, s...","Alright, get ready for a deep dive into the do...",None,"The therapeutic approaches outlined in PART 2,...",True,True,True,6.234645,ccd8670f-ec2c-49b6-b143-2d65594080d5,019c689e-8fa4-7933-b44a-02a28c77c6cb
4,"How can u use the habit loop: cue, routine, re...","Alright, let’s crank up the dopeness on buildi...",None,"To use the habit loop—cue, routine, reward—to ...",True,True,True,5.020893,7fb5c52a-6a9f-4af1-abcb-3599883a52e3,019c689e-cadd-7bb1-9f5c-beb06e44d4dc
5,How can meal planning and preparation help me ...,"Alright, let’s crank that wellness dial all th...",None,Meal planning and preparation are essential fo...,True,True,True,5.151108,b5806079-4d65-4b1d-9837-a11f2a5929dd,019c689f-0552-79d3-832c-63c7e07f0aef
6,How can mindfulness and meditation practices i...,"Alright, strap in for a mind-melting fusion of...",None,Mindfulness and meditation practices can incor...,True,True,True,5.386945,65121dbb-7fdb-4e7f-9a84-b54e498e62d9,019c689f-44d6-7c92-ba32-ed7e6750cc97
7,How do practices related to sleep hygiene and ...,"Oh heck yes, let’s dive into this! Sleep hygie...",None,"Practicing good sleep hygiene, such as maintai...",True,True,True,4.848484,36cf455e-572f-4623-85b0-994f4f16342a,019c689f-99e7-7fa3-93c7-e4dc1e4dc785
8,How do psychologists contribute to managing di...,"Alright, let's crank up the rad vibes and lase...",None,Psychologists specialize in therapy and assess...,False,False,False,3.743734,5cadf8cd-c2d2-40f6-bf3f-fb3dd779e808,019c689f-dec1-75b3-8166-64cecee398eb
9,How do B vitamins help in maintaining good men...,"Alright, here’s the skinny on B vitamins and y...",None,"B vitamins are found in whole grains, eggs, an...",True,True,True,2.947420,1404ae5a-f868-4e84-b478-5cf15e57dc77,019c68a0-10f5-7042-aef0-631c4121fafc


---
## 🏗️ Activity #2: Analyze Evaluation Results

Provide a screenshot of the difference between the two chains in LangSmith, and explain why you believe certain metrics changed in certain ways.

##### Answer:

**Default Eval:**
![Default Eval](screenshots/default%20eval.png)

**Dope Eval:**
![Dope Eval](screenshots/dope%20eval.png)

**Dopeness**: This of course saw the biggest improvement on the dope chain — the default chain scored mostly 0s while the dope chain scored mostly 1s. This makes sense because the dope prompt directly tells the LLM to be "rad" and avoid generic responses, directly aiming at improving what this evaluator measures.

**Helpfulness**: This slightly improved in the dope chain. This is likely due to the combination of larger chunks (1000 vs 500) and the more powerful `text-embedding-3-large` model, both of which improve retrieval quality and provide better context for generating helpful answers.

**QA correctness**: This also slightly improved in the dope chain, also likely due to better retrieval from the larger embedding model and chunk size. However, the dope prompt's emphasis on style potentially sacrifices precision for flair, which could have prevented further retrieval gains.

---
## Summary

In this session, we:

1. **Generated synthetic test data** using Ragas' knowledge graph-based approach
2. **Explored query synthesizers** for creating diverse question types
3. **Loaded synthetic data** into a LangSmith dataset for evaluation
4. **Built and evaluated a RAG chain** using LangSmith evaluators
5. **Iterated on the pipeline** by modifying chunk size, embedding model, and prompt — then measured the impact

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **LangSmith evaluators** enable systematic comparison of pipeline versions
- **Small changes matter** — chunk size, embedding model, and prompt modifications can significantly affect evaluation scores